In [8]:
import pandas as pd
import numpy as np
import cv2
import os
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv3D, MaxPooling3D, Flatten, Dense, GlobalAveragePooling3D, LSTM, TimeDistributed


In [9]:
def extract_frame_sequence(video_path, start_frame, end_frame):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Unable to open video file {video_path}")
        return None

    frame_list = []
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

    while start_frame <= end_frame:
        ret, frame = cap.read()
        if not ret:
            break
        frame_list.append(frame)
        start_frame += 1

    cap.release()
    
    if len(frame_list) == 0:
        return None  # Skip if no frames were extracted
    
    return frame_list


In [10]:
def preprocess_frames(frames, target_size=(128, 128), max_frames=100):
    processed_frames = []

    for frame in frames:
        resized_frame = cv2.resize(frame, target_size)
        gray_frame = cv2.cvtColor(resized_frame, cv2.COLOR_BGR2GRAY)
        gray_frame = gray_frame.astype('float32') / 255.0
        gray_frame = np.expand_dims(gray_frame, axis=-1)  # Add channel dimension
        processed_frames.append(gray_frame)

    # Pad or truncate frames to have a fixed length
    if len(processed_frames) < max_frames:
        # Pad with zeros
        while len(processed_frames) < max_frames:
            processed_frames.append(np.zeros((target_size[0], target_size[1], 1), dtype='float32'))
    else:
        # Truncate
        processed_frames = processed_frames[:max_frames]

    return np.array(processed_frames, dtype='float32')


In [11]:
# Load the CSV file into a DataFrame for the fall detection dataset
df_fall = pd.read_csv(r'C:\Users\Sahitya A\Downloads\archive (18)\data_tuple3.csv')

# Check if the data is loaded correctly
print(df_fall.head())

# Define base path for the violence detection dataset
base_path_violence = r"C:\Users\Sahitya A\Desktop\fall-detection(mini-project\violence-detection-dataset"

def frame_sequence_generator():
    # For the fall detection dataset
    for index, row in df_fall.iterrows():
        chute = row['chute']
        cam = int(row['cam'])
        start = int(row['start'])
        end = int(row['end'])
        label = int(row['label'])

        chute_folder = f'chute{int(chute):02d}'  
        video_path = f'C:\\Users\\Sahitya A\\Downloads\\archive (18)\\dataset\\dataset\\{chute_folder}\\cam{cam}.avi'

        frames = extract_frame_sequence(video_path, start, end)
        if frames is not None:
            preprocessed_frames = preprocess_frames(frames)
            yield preprocessed_frames, label 

    # For the violence detection dataset
    for category in ["non-violent", "violent"]:
        for cam in ["cam1", "cam2"]:
            cam_path = os.path.join(base_path_violence, category, cam)
            for video_file in os.listdir(cam_path):
                if video_file.endswith('.mp4'):
                    video_path = os.path.join(cam_path, video_file)
                    frames = extract_frame_sequence(video_path, 0, 100)  # Example: extract first 100 frames
                    label = 0 if category == "non-violent" else 1  # Labeling: 0 for non-violent, 1 for violent
                    
                    if frames is not None:
                        preprocessed_frames = preprocess_frames(frames)
                        yield preprocessed_frames, label


   chute  cam   start     end  label
0    1.0  1.0  1052.0  1082.0    0.0
1    1.0  1.0  1083.0  1113.0    1.0
2    1.0  1.0  1114.0  1144.0    0.0
3    1.0  2.0  1052.0  1082.0    0.0
4    1.0  2.0  1083.0  1113.0    1.0


In [12]:
# Define the dataset from the generator
dataset = tf.data.Dataset.from_generator(
    frame_sequence_generator,
    output_signature=(
        tf.TensorSpec(shape=(100, 128, 128, 1), dtype=tf.float32),  # Fixed shape
        tf.TensorSpec(shape=(), dtype=tf.int32)
    )
)

# Shuffle and batch the dataset
batch_size = 8
dataset = dataset.shuffle(buffer_size=1000).batch(batch_size).prefetch(buffer_size=tf.data.AUTOTUNE)

# Estimate the size of the dataset for splitting
train_size = int(0.8 * len(df_fall))  # Adjust as necessary for the violence dataset
train_dataset = dataset.take(train_size)
val_dataset = dataset.skip(train_size)

steps_per_epoch = train_size // batch_size



In [13]:
# Define the 3D CNN model
model = Sequential([
    Conv3D(16, (3, 3, 3), activation='relu', input_shape=(100, 128, 128, 1)),  # Fixed input shape
    MaxPooling3D(pool_size=(2, 2, 2)),
    Conv3D(32, (3, 3, 3), activation='relu'),
    MaxPooling3D(pool_size=(2, 2, 2)),
    
    TimeDistributed(Flatten()),  # Flatten each time step's output
    LSTM(64, return_sequences=False),  # LSTM layer for sequence processing
    
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')  # Binary classification: fall (1) or no fall (0)
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


In [55]:
# Train the model
epochs = 1  # Adjust the number of epochs as needed
history = model.fit(train_dataset, validation_data=val_dataset, epochs=epochs,steps_per_epoch=steps_per_epoch)


Error: Unable to open video file C:\Users\Sahitya A\Downloads\archive (18)\dataset\dataset\chute23\cam55.avi
55/55 [==============================] - 1519s 22s/step - loss: 0.6753 - accuracy: 0.5864


In [56]:
# Save the trained model
model.save('fall_and_violence_detection_3dcnn_model.h5')



In [22]:
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
import os

# Load the trained model
model = load_model('fall_and_violence_detection_3dcnn_model.h5')

# Helper functions for frame extraction and preprocessing
def extract_frames(video_path, start_frame, num_frames=100, target_size=(128, 128)):
    cap = cv2.VideoCapture(video_path)
    frames = []
    
    # Set the starting frame
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    
    for _ in range(num_frames):
        ret, frame = cap.read()
        if not ret:
            break
        resized_frame = cv2.resize(frame, target_size)
        gray_frame = cv2.cvtColor(resized_frame, cv2.COLOR_BGR2GRAY)
        gray_frame = gray_frame.astype('float32') / 255.0
        frames.append(gray_frame)
    
    cap.release()
    
    # Add channel dimension
    frames = np.array(frames)
    frames = np.expand_dims(frames, axis=-1)
    
    # Pad or truncate to ensure fixed length
    if len(frames) < num_frames:
        padding = np.zeros((num_frames - len(frames), target_size[0], target_size[1], 1), dtype='float32')
        frames = np.concatenate((frames, padding), axis=0)
    else:
        frames = frames[:num_frames]
    
    return np.expand_dims(frames, axis=0)  # Add batch dimension

def save_video_with_alerts(input_video_path, output_video_path, segment_length=100, threshold=0.5):
    cap = cv2.VideoCapture(input_video_path)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Define the codec and create VideoWriter object to save the output video
    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))
    
    # Process the video in segments
    for start_frame in range(0, frame_count, segment_length):
        frames = extract_frames(input_video_path, start_frame, segment_length)
        if frames is None:
            continue
        
        # Predict abnormal activity probability
        abnormal_probability = model.predict(frames)[0][0]
        
        for i in range(segment_length):
            # Go to the specific frame to overlay the message
            cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame + i)
            ret, frame = cap.read()
            if not ret:
                break
            
            # Overlay alert if abnormal activity detected
            if abnormal_probability >= threshold:
                cv2.putText(frame, "ALERT: Abnormal Activity Detected!", (50, 50), 
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

            # Write the frame with the alert to the output video
            out.write(frame)
    
    cap.release()
    out.release()
    print(f"Output saved to {output_video_path}")

# Test the model on a sample video and save the output
sample_video_path = r'C:\Users\Sahitya A\Desktop\fall-detection(mini-project\violence-detection-dataset\violent\cam1\15.mp4'
output_video_path = r'C:\Users\Sahitya A\Desktop\fall-detection-output.avi'
save_video_with_alerts(sample_video_path, output_video_path, segment_length=100, threshold=0.5)


1/1 [==============================] - 0s 231ms/step
Output saved to C:\Users\Sahitya A\Desktop\fall-detection-output.avi


In [25]:
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
import os

# Load the trained model
model = load_model('fall_and_violence_detection_3dcnn_model.h5')

# Helper functions for frame extraction and preprocessing
def extract_frames(video_path, start_frame, num_frames=100, target_size=(128, 128)):
    cap = cv2.VideoCapture(video_path)
    frames = []
    
    # Set the starting frame
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    
    for _ in range(num_frames):
        ret, frame = cap.read()
        if not ret:
            break
        resized_frame = cv2.resize(frame, target_size)
        gray_frame = cv2.cvtColor(resized_frame, cv2.COLOR_BGR2GRAY)
        gray_frame = gray_frame.astype('float32') / 255.0
        frames.append(gray_frame)
    
    cap.release()
    
    # Add channel dimension
    frames = np.array(frames)
    frames = np.expand_dims(frames, axis=-1)
    
    # Pad or truncate to ensure fixed length
    if len(frames) < num_frames:
        padding = np.zeros((num_frames - len(frames), target_size[0], target_size[1], 1), dtype='float32')
        frames = np.concatenate((frames, padding), axis=0)
    else:
        frames = frames[:num_frames]
    
    return np.expand_dims(frames, axis=0)  # Add batch dimension

def save_video_with_alerts(input_video_path, output_video_path, segment_length=100, threshold=0.5):
    cap = cv2.VideoCapture(input_video_path)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Define the codec and create VideoWriter object to save the output video
    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))
    
    # Process the video in segments
    for start_frame in range(0, frame_count, segment_length):
        frames = extract_frames(input_video_path, start_frame, segment_length)
        if frames is None:
            continue
        
        # Predict abnormal activity probability
        abnormal_probability = model.predict(frames)[0][0]
        
        for i in range(segment_length):
            # Go to the specific frame to overlay the message
            cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame + i)
            ret, frame = cap.read()
            if not ret:
                break
            
            # Overlay alert if abnormal activity detected
            if abnormal_probability >= threshold:
                cv2.putText(frame, "ALERT: Abnormal Activity Detected!", (50, 50), 
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

            # Write the frame with the alert (or without if no alert) to the output video
            out.write(frame)
    
    cap.release()
    out.release()
    print(f"Output saved to {output_video_path}")

# Test the model on a sample video and save the output
sample_video_path = r'C:\Users\Sahitya A\Desktop\fall-detection(mini-project\dataset\chute01\cam2.avi'
output_video_path = r'C:\Users\Sahitya A\Desktop\fall-detection-output1.avi'
save_video_with_alerts(sample_video_path, output_video_path, segment_length=100, threshold=0.5)


1/1 [==============================] - 0s 204ms/step
Output saved to C:\Users\Sahitya A\Desktop\fall-detection-output1.avi
